# 3장 · 자료 다루기

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [3장 · 자료 다루기](https://grow.minds.kr/textbooks/css-methods/causal/book/ch03-자료-다루기.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 정제: 몇 명이 남나
주의 점검 통과자만 남깁니다. 380→366, 590→566.

In [ ]:
exp = load("exp"); svy = load("svy")
print("실험판:", 380, "->", len(exp), " 설문판:", 590, "->", len(svy))

## 3. 파생변수 검산
21문항 평균을 직접 만들어, 저장된 hjs와 전원 일치하는지 두 경로로 확인.

In [ ]:
items = [c for c in svy.columns if c.startswith("hjs_") and c != "hjs"]
print("1번 응답자 합÷21 =", round(svy[items].iloc[0].sum()/21, 3))   # 4.571
print("내가 만든 열과 저장된 열의 최대 차이 =", round((svy[items].mean(axis=1)-svy.hjs).abs().max(), 4))

## 4. long과 wide, 그리고 결측
패널판을 wide로 펼치면 이탈이 결측으로 드러납니다(2차 80, 3차 150).

In [ ]:
panel = pd.read_csv("data/journey_panel.csv")
w = panel.pivot(index="id", columns="wave", values="mil")
print("2차 결측:", int(w[2].isna().sum()), " 3차 결측:", int(w[3].isna().sum()))

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.